# Phase 16 — Pipeline Audit + Strong Cross-Dataset Baselines (FIXED)

**NetShield — performance-first pipeline audit**

This notebook replaces the previous Phase 16 loading/audit logic.

### What is fixed
- **UNSW-NB15** is loaded normally using the official train/test split.
- **CICIDS2017 is NEVER concatenated into one giant dataframe.** It is scanned and sampled in chunks, so the notebook does not require a second ~500 MB+ dataframe copy.
- No `cicids` full-dataframe variable is required.
- The audit does not call `select_dtypes(...).copy()` on millions of rows.
- CICIDS model features are read with `usecols` only.
- Binary labels are created explicitly and verified.
- Cross-dataset evaluation uses the same frozen CICIDS evaluation sample for all candidate models/representations.
- Target CICIDS labels are used **only for final evaluation**, never for feature construction, preprocessing fitting, model selection, or threshold selection.

### Performance goal
The immediate goal is to establish a **sound, stronger baseline** before adding another adaptation method. We compare the verified 11-feature semantic representation against a richer deterministic expansion and compare strong tabular models.

Primary metrics: **F1, Macro-F1, PR-AUC, ROC-AUC, recall, FPR**. Accuracy is also reported, but it is not the primary metric because of class imbalance.


In [ ]:
# ============================================================
# 0. Imports, paths, reproducibility
# ============================================================

from pathlib import Path
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
CICIDS_CHUNK_SIZE = 100_000
CICIDS_SAMPLE_N = 1_000_000
CICIDS_EVAL_N = 300_000

# Try to locate the project automatically.
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        if (base / "data" / "raw" / "UNSW-NB15").exists() and (base / "data" / "raw" / "CICIDS2017").exists():
            return base
    # Common case when notebook is executed from project/notebooks
    if Path.cwd().name.lower() == "notebooks":
        return Path.cwd().parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
UNSW_ROOT = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
CICIDS_ROOT = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"
OUT = PROJECT_ROOT / "outputs" / "phase16_pipeline_audit_fixed"
TABLES = OUT / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UNSW_ROOT   :", UNSW_ROOT)
print("CICIDS_ROOT :", CICIDS_ROOT)
print("OUTPUT      :", OUT)


## 1. Load UNSW and discover CICIDS files

The previous notebook loaded every CICIDS CSV into `parts` and then called `pd.concat`. That was the source of the out-of-memory failures. Here we keep UNSW in memory and only keep the **list of CICIDS files**.

In [ ]:
# ============================================================
# 1. UNSW normal load + CICIDS file discovery
# ============================================================

u_train = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_training-set.csv"
u_test = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_testing-set.csv"

if not u_train.exists():
    matches = list(UNSW_ROOT.rglob("UNSW_NB15_training-set.csv"))
    if not matches:
        raise FileNotFoundError("Could not find UNSW_NB15_training-set.csv")
    u_train = matches[0]

if not u_test.exists():
    matches = list(UNSW_ROOT.rglob("UNSW_NB15_testing-set.csv"))
    if not matches:
        raise FileNotFoundError("Could not find UNSW_NB15_testing-set.csv")
    u_test = matches[0]

print("Loading UNSW...")
unsw_train = pd.read_csv(u_train)
unsw_test = pd.read_csv(u_test)

if "label" not in unsw_train.columns or "label" not in unsw_test.columns:
    raise KeyError("UNSW train/test must contain a 'label' column.")

unsw_train["label_binary"] = pd.to_numeric(unsw_train["label"], errors="coerce").astype("int8")
unsw_test["label_binary"] = pd.to_numeric(unsw_test["label"], errors="coerce").astype("int8")

cicids_files = sorted(CICIDS_ROOT.rglob("*.csv"))
if not cicids_files:
    raise FileNotFoundError(f"No CICIDS CSV files found under {CICIDS_ROOT}")

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)
print("\nCICIDS files:", len(cicids_files))
for p in cicids_files:
    print(" -", p.name)


## 2. Memory-safe CICIDS data-quality audit

This scans CICIDS in chunks. It never creates a 2.8M-row dataframe. We count rows, missing values, and binary label totals per file.

In [ ]:
# ============================================================
# 2. Memory-safe CICIDS audit
# ============================================================

cicids_audit_rows = []

for path in cicids_files:
    print(f"Auditing: {path.name}")
    rows = 0
    missing_cells = 0
    benign = 0
    attack = 0

    for chunk in pd.read_csv(path, chunksize=CICIDS_CHUNK_SIZE, low_memory=True):
        chunk.columns = chunk.columns.str.strip()
        if "Label" not in chunk.columns:
            raise KeyError(f"'Label' not found in {path.name}")

        rows += len(chunk)
        missing_cells += int(chunk.isna().sum().sum())

        labels = chunk["Label"].astype(str).str.strip().str.upper()
        benign += int((labels == "BENIGN").sum())
        attack += int((labels != "BENIGN").sum())

        del labels, chunk

    cicids_audit_rows.append({
        "file": path.name,
        "rows": rows,
        "missing_cells": missing_cells,
        "benign": benign,
        "attack": attack,
        "attack_rate": attack / rows if rows else np.nan,
    })
    gc.collect()

cicids_audit = pd.DataFrame(cicids_audit_rows)

display(cicids_audit)

TOTAL_CICIDS_ROWS = int(cicids_audit["rows"].sum())
TOTAL_CICIDS_BENIGN = int(cicids_audit["benign"].sum())
TOTAL_CICIDS_ATTACK = int(cicids_audit["attack"].sum())

print("\nCICIDS total rows :", TOTAL_CICIDS_ROWS)
print("CICIDS benign     :", TOTAL_CICIDS_BENIGN)
print("CICIDS attack     :", TOTAL_CICIDS_ATTACK)
print("CICIDS attack rate:", round(TOTAL_CICIDS_ATTACK / TOTAL_CICIDS_ROWS, 4))

cicids_audit.to_csv(TABLES / "01_cicids_chunked_audit.csv", index=False)

unsw_audit = pd.DataFrame([
    {
        "dataset": "UNSW_train",
        "rows": len(unsw_train),
        "columns": unsw_train.shape[1],
        "numeric_columns": sum(pd.api.types.is_numeric_dtype(unsw_train[c]) for c in unsw_train.columns),
        "missing_cells": int(unsw_train.isna().sum().sum()),
        "memory_MB": round(unsw_train.memory_usage(deep=True).sum() / 1024**2, 2),
    },
    {
        "dataset": "UNSW_test",
        "rows": len(unsw_test),
        "columns": unsw_test.shape[1],
        "numeric_columns": sum(pd.api.types.is_numeric_dtype(unsw_test[c]) for c in unsw_test.columns),
        "missing_cells": int(unsw_test.isna().sum().sum()),
        "memory_MB": round(unsw_test.memory_usage(deep=True).sum() / 1024**2, 2),
    },
])

display(unsw_audit)
unsw_audit.to_csv(TABLES / "02_unsw_audit.csv", index=False)

print("\nUNSW train labels:")
display(unsw_train["label_binary"].value_counts(dropna=False).rename("count").to_frame())
print("UNSW test labels:")
display(unsw_test["label_binary"].value_counts(dropna=False).rename("count").to_frame())


## 3. Verify the exact CICIDS feature schema without loading the dataset

Only the columns needed for the verified semantic representation are read. This avoids memory waste and also makes the cross-dataset mapping explicit.

In [ ]:
# ============================================================
# 3. Schema verification
# ============================================================

CICIDS_REQUIRED = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Average Packet Size",
    "Label",
]

# Read only the header of the first file.
header = pd.read_csv(cicids_files[0], nrows=0)
header.columns = header.columns.str.strip()
missing_required = [c for c in CICIDS_REQUIRED if c not in header.columns]

if missing_required:
    raise KeyError(
        "Required CICIDS columns missing from the first file: " + str(missing_required)
    )

print("All required CICIDS semantic columns are present.")
print(CICIDS_REQUIRED)

print("\nUNSW required columns:")
print(["dur", "spkts", "dpkts", "sbytes", "dbytes", "label"])

missing_unsw = [c for c in ["dur", "spkts", "dpkts", "sbytes", "dbytes", "label"] if c not in unsw_train.columns]
if missing_unsw:
    raise KeyError("Required UNSW columns missing: " + str(missing_unsw))


## 4. Chunked CICIDS sampling — no giant `cicids` dataframe

We need enough CICIDS data for a meaningful strong baseline, but we do not need all 2.8M rows resident in RAM. The sampler performs two passes:

1. count benign/attack rows globally;
2. sample approximately the requested number from each class across all files.

The sampled dataframe contains only the semantic source columns plus the binary label. This is much smaller than the raw 24+ column dataset.

In [ ]:
# ============================================================
# 4A. Global class counts for sampling
# ============================================================

global_counts = {0: 0, 1: 0}

for path in cicids_files:
    for chunk in pd.read_csv(
        path,
        usecols=["Label"],
        chunksize=CICIDS_CHUNK_SIZE,
        low_memory=True,
    ):
        labels = chunk["Label"].astype(str).str.strip().str.upper()
        global_counts[0] += int((labels == "BENIGN").sum())
        global_counts[1] += int((labels != "BENIGN").sum())
        del labels, chunk
    gc.collect()

print("Global CICIDS class counts:", global_counts)
print("Global attack rate:", global_counts[1] / sum(global_counts.values()))


In [ ]:
# ============================================================
# 4B. Chunked stratified sampler
# ============================================================

# We sample CICIDS_SAMPLE_N rows, preserving the full-dataset benign/attack prior.
# The final sampled pool is split into a model-training portion and a frozen
# cross-dataset evaluation portion.

sample_n = min(CICIDS_SAMPLE_N, TOTAL_CICIDS_ROWS)
if sample_n < 100_000:
    raise ValueError("CICIDS sample is unexpectedly small.")

rng = np.random.default_rng(RANDOM_STATE)

target_quota = {
    0: int(round(sample_n * global_counts[0] / TOTAL_CICIDS_ROWS)),
    1: int(round(sample_n * global_counts[1] / TOTAL_CICIDS_ROWS)),
}
# Fix rounding exactly.
target_quota[1] = sample_n - target_quota[0]

print("Target sample size:", sample_n)
print("Target class quotas:", target_quota)

# Allocate each file's contribution proportional to its class counts.
file_counts = cicids_audit.set_index("file").to_dict("index")

sample_parts = []

for path in cicids_files:
    meta = file_counts[path.name]
    n_file = int(meta["rows"])

    # Per-file quotas for each class. Rounding is corrected later by the
    # final concat + global resampling step.
    q0 = int(round(target_quota[0] * meta["benign"] / global_counts[0])) if global_counts[0] else 0
    q1 = int(round(target_quota[1] * meta["attack"] / global_counts[1])) if global_counts[1] else 0

    if q0 == 0 and q1 == 0:
        continue

    usecols = [
        "Flow Duration",
        "Total Fwd Packets",
        "Total Backward Packets",
        "Total Length of Fwd Packets",
        "Total Length of Bwd Packets",
        "Flow Packets/s",
        "Fwd Packets/s",
        "Bwd Packets/s",
        "Average Packet Size",
        "Label",
    ]

    file_parts = []

    for chunk in pd.read_csv(
        path,
        usecols=usecols,
        chunksize=CICIDS_CHUNK_SIZE,
        low_memory=True,
    ):
        chunk.columns = chunk.columns.str.strip()
        labels = chunk["Label"].astype(str).str.strip().str.upper()
        chunk["label_binary"] = (labels != "BENIGN").astype("int8")

        benign_part = chunk[chunk["label_binary"] == 0]
        attack_part = chunk[chunk["label_binary"] == 1]

        # Allocate this chunk's share according to how much of the file remains.
        # A random chunk sample is enough because we already stratified by class
        # and are sampling across every file.
        if len(benign_part):
            take0 = min(len(benign_part), max(0, int(round(q0 * len(benign_part) / max(meta["benign"], 1)))))
            if take0:
                file_parts.append(benign_part.sample(n=take0, random_state=RANDOM_STATE))
        if len(attack_part):
            take1 = min(len(attack_part), max(0, int(round(q1 * len(attack_part) / max(meta["attack"], 1)))))
            if take1:
                file_parts.append(attack_part.sample(n=take1, random_state=RANDOM_STATE))

        del labels, benign_part, attack_part, chunk

    if file_parts:
        sample_parts.append(pd.concat(file_parts, ignore_index=True))
    del file_parts
    gc.collect()

cicids_sample = pd.concat(sample_parts, ignore_index=True)
del sample_parts

# Enforce exact total sample size while keeping the natural binary prior as closely as possible.
if len(cicids_sample) > sample_n:
    cicids_sample = cicids_sample.sample(n=sample_n, random_state=RANDOM_STATE).reset_index(drop=True)
elif len(cicids_sample) < sample_n:
    print(f"Warning: requested {sample_n} but obtained {len(cicids_sample)} rows due to per-file rounding.")

print("Final CICIDS sampled rows:", len(cicids_sample))
print("Sample labels:")
display(cicids_sample["label_binary"].value_counts().rename("count").to_frame())


## 5. Build the verified common semantic representation

These are the same 11 quantities used in the earlier phases. No new questionable feature mapping is introduced here.

In [ ]:
# ============================================================
# 5. Common semantic representation
# ============================================================

COMMON_11 = [
    "flow_duration_sec",
    "fwd_packets",
    "bwd_packets",
    "fwd_bytes",
    "bwd_bytes",
    "total_packets",
    "total_bytes",
    "packet_rate",
    "fwd_packet_rate",
    "bwd_packet_rate",
    "avg_packet_size",
]


def safe_divide(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return a.div(b.replace(0, np.nan))


def build_unsw_common(df):
    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce")
    x["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce")
    x["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce")
    x["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce")
    x["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce")
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = safe_divide(x["total_packets"], x["flow_duration_sec"])
    x["fwd_packet_rate"] = safe_divide(x["fwd_packets"], x["flow_duration_sec"])
    x["bwd_packet_rate"] = safe_divide(x["bwd_packets"], x["flow_duration_sec"])
    x["avg_packet_size"] = safe_divide(x["total_bytes"], x["total_packets"])
    return x.replace([np.inf, -np.inf], np.nan).astype("float32")


def build_cicids_common(df):
    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = safe_divide(df["Flow Duration"], 1_000_000)
    x["fwd_packets"] = pd.to_numeric(df["Total Fwd Packets"], errors="coerce")
    x["bwd_packets"] = pd.to_numeric(df["Total Backward Packets"], errors="coerce")
    x["fwd_bytes"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce")
    x["bwd_bytes"] = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce")
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = pd.to_numeric(df["Flow Packets/s"], errors="coerce")
    x["fwd_packet_rate"] = pd.to_numeric(df["Fwd Packets/s"], errors="coerce")
    x["bwd_packet_rate"] = pd.to_numeric(df["Bwd Packets/s"], errors="coerce")
    x["avg_packet_size"] = pd.to_numeric(df["Average Packet Size"], errors="coerce")
    return x.replace([np.inf, -np.inf], np.nan).astype("float32")


X_unsw_train_11 = build_unsw_common(unsw_train)
X_unsw_test_11 = build_unsw_common(unsw_test)
X_cicids_11 = build_cicids_common(cicids_sample)

y_unsw_train = unsw_train["label_binary"].to_numpy(dtype="int8")
y_unsw_test = unsw_test["label_binary"].to_numpy(dtype="int8")
y_cicids = cicids_sample["label_binary"].to_numpy(dtype="int8")

print("UNSW train 11:", X_unsw_train_11.shape)
print("UNSW test 11 :", X_unsw_test_11.shape)
print("CICIDS 11    :", X_cicids_11.shape)


## 6. Rich semantic expansion

The expansion is deterministic from the verified 11 quantities. It adds directional fractions/ratios, bytes-per-packet, byte rates, and log transforms. It does **not** use CICIDS labels.

In [ ]:
# ============================================================
# 6. Expanded semantic representation
# ============================================================

def build_expanded_semantic(x11):
    x = x11.copy()
    eps = 1e-9

    x["fwd_packet_fraction"] = x["fwd_packets"] / (x["total_packets"] + eps)
    x["bwd_packet_fraction"] = x["bwd_packets"] / (x["total_packets"] + eps)
    x["fwd_byte_fraction"] = x["fwd_bytes"] / (x["total_bytes"] + eps)
    x["bwd_byte_fraction"] = x["bwd_bytes"] / (x["total_bytes"] + eps)
    x["fwd_bwd_packet_ratio"] = x["fwd_packets"] / (x["bwd_packets"] + eps)
    x["fwd_bwd_byte_ratio"] = x["fwd_bytes"] / (x["bwd_bytes"] + eps)
    x["fwd_bytes_per_packet"] = x["fwd_bytes"] / (x["fwd_packets"] + eps)
    x["bwd_bytes_per_packet"] = x["bwd_bytes"] / (x["bwd_packets"] + eps)
    x["bytes_per_packet"] = x["total_bytes"] / (x["total_packets"] + eps)

    safe_duration = x["flow_duration_sec"].replace(0, np.nan)
    x["byte_rate"] = x["total_bytes"] / safe_duration
    x["fwd_byte_rate"] = x["fwd_bytes"] / safe_duration
    x["bwd_byte_rate"] = x["bwd_bytes"] / safe_duration

    log_cols = [
        "flow_duration_sec", "fwd_packets", "bwd_packets", "fwd_bytes", "bwd_bytes",
        "total_packets", "total_bytes", "packet_rate", "fwd_packet_rate",
        "bwd_packet_rate", "avg_packet_size", "byte_rate", "fwd_byte_rate", "bwd_byte_rate",
    ]
    for c in log_cols:
        x[f"log1p_{c}"] = np.log1p(x[c].clip(lower=0))

    return x.replace([np.inf, -np.inf], np.nan).astype("float32")

X_unsw_train_exp = build_expanded_semantic(X_unsw_train_11)
X_unsw_test_exp = build_expanded_semantic(X_unsw_test_11)
X_cicids_exp = build_expanded_semantic(X_cicids_11)

print("11 features:", X_unsw_train_11.shape[1])
print("Expanded features:", X_unsw_train_exp.shape[1])
print(list(X_unsw_train_exp.columns))


## 7. Frozen cross-dataset evaluation design

For **UNSW → CICIDS**, UNSW official training data are split into source-fit/source-validation. The CICIDS sampled pool is split into a frozen target evaluation set and a CICIDS training portion used only for the reverse direction.

For **CICIDS → UNSW**, the CICIDS training portion is split into source-fit/source-validation, and the official UNSW test set is the frozen target evaluation set.

Thresholds are selected only on the source validation set. No CICIDS target labels are used to select a model or threshold.

In [ ]:
# ============================================================
# 7. Source/target splits
# ============================================================

# UNSW source fit/validation
unsw_fit_idx, unsw_val_idx = train_test_split(
    np.arange(len(y_unsw_train)),
    test_size=0.20,
    stratify=y_unsw_train,
    random_state=RANDOM_STATE,
)

# CICIDS sampled pool -> source fit, source validation, and a separate
# frozen target evaluation set. Target evaluation rows are NEVER reused
# for source validation.
cic_source_pool_idx, cic_eval_idx = train_test_split(
    np.arange(len(y_cicids)),
    test_size=min(CICIDS_EVAL_N, max(10_000, int(0.30 * len(y_cicids)))),
    stratify=y_cicids,
    random_state=RANDOM_STATE,
)

cic_fit_idx, cic_val_idx = train_test_split(
    cic_source_pool_idx,
    test_size=0.20,
    stratify=y_cicids[cic_source_pool_idx],
    random_state=RANDOM_STATE,
)

print("UNSW source fit:", len(unsw_fit_idx))
print("UNSW source val:", len(unsw_val_idx))
print("CICIDS source fit:", len(cic_fit_idx))
print("CICIDS source val:", len(cic_val_idx))
print("CICIDS frozen eval:", len(cic_eval_idx))

y_cic_eval = y_cicids[cic_eval_idx]


## 8. Leakage-safe preprocessing + threshold selection

Tree models do not require scaling. We use median imputation fitted on the **source-fit data only**. Thresholds are selected by source-validation F1, then frozen for the target dataset.

In [ ]:
# ============================================================
# 8. Utilities
# ============================================================

def fit_transform_source_target(X_source_fit, X_source_val, X_target):
    imp = SimpleImputer(strategy="median")
    Xf = imp.fit_transform(X_source_fit)
    Xv = imp.transform(X_source_val)
    Xt = imp.transform(X_target)
    return Xf, Xv, Xt, imp


def choose_threshold(y_true, proba):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [f1_score(y_true, proba >= t, zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(scores))])


def metrics_row(y_true, proba, threshold, direction, representation, model_name):
    pred = (proba >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "direction": direction,
        "representation": representation,
        "model": model_name,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "predicted_attack_rate": float(pred.mean()),
    }


def make_models():
    return {
        "RandomForest": RandomForestClassifier(
            n_estimators=250,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=250,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        ),
    }


## 9. Experiment A — UNSW → CICIDS

This is the main cross-dataset performance test. We compare the compact semantic representation and the richer semantic representation using the same frozen CICIDS evaluation set.

In [ ]:
# ============================================================
# 9. UNSW -> CICIDS
# ============================================================

experiments = []

representation_sets = {
    "11-semantic": (X_unsw_train_11, X_cicids_11),
    "expanded-semantic": (X_unsw_train_exp, X_cicids_exp),
}

for rep_name, (X_source_all, X_target_all) in representation_sets.items():
    print(f"\n===== {rep_name}: UNSW -> CICIDS =====")

    X_source_fit_raw = X_source_all.iloc[unsw_fit_idx]
    X_source_val_raw = X_source_all.iloc[unsw_val_idx]
    X_target_raw = X_target_all.iloc[cic_eval_idx]

    Xf, Xv, Xt, imp = fit_transform_source_target(
        X_source_fit_raw,
        X_source_val_raw,
        X_target_raw,
    )

    yfit = y_unsw_train[unsw_fit_idx]
    yval = y_unsw_train[unsw_val_idx]

    for model_name, model in make_models().items():
        start = time.perf_counter()
        model.fit(Xf, yfit)
        train_seconds = time.perf_counter() - start

        val_proba = model.predict_proba(Xv)[:, 1]
        threshold = choose_threshold(yval, val_proba)

        target_proba = model.predict_proba(Xt)[:, 1]

        row = metrics_row(
            y_cic_eval,
            target_proba,
            threshold,
            "UNSW_to_CICIDS",
            rep_name,
            model_name,
        )
        row["train_seconds"] = train_seconds
        experiments.append(row)

        print(
            model_name,
            "threshold=", round(threshold, 3),
            "F1=", round(row["f1"], 4),
            "PR-AUC=", round(row["pr_auc"], 4),
            "time(s)=", round(train_seconds, 1),
        )

res_unsw_cic = pd.DataFrame(experiments)
display(res_unsw_cic.sort_values("f1", ascending=False))


## 10. Experiment B — CICIDS → UNSW

The reverse direction uses the chunked CICIDS sample for source training and validation, while the official UNSW test set is the frozen target evaluation set.

In [ ]:
# ============================================================
# 10. CICIDS -> UNSW
# ============================================================

reverse_experiments = []

representation_sets_reverse = {
    "11-semantic": (X_cicids_11, X_unsw_test_11),
    "expanded-semantic": (X_cicids_exp, X_unsw_test_exp),
}

for rep_name, (X_source_all, X_target_all) in representation_sets_reverse.items():
    print(f"\n===== {rep_name}: CICIDS -> UNSW =====")

    X_source_fit_raw = X_source_all.iloc[cic_fit_idx]
    X_source_val_raw = X_source_all.iloc[cic_val_idx]
    X_target_raw = X_target_all

    Xf, Xv, Xt, imp = fit_transform_source_target(
        X_source_fit_raw,
        X_source_val_raw,
        X_target_raw,
    )

    yfit = y_cicids[cic_fit_idx]
    yval = y_cicids[cic_val_idx]

    for model_name, model in make_models().items():
        start = time.perf_counter()
        model.fit(Xf, yfit)
        train_seconds = time.perf_counter() - start

        val_proba = model.predict_proba(Xv)[:, 1]
        threshold = choose_threshold(yval, val_proba)
        target_proba = model.predict_proba(Xt)[:, 1]

        row = metrics_row(
            y_unsw_test,
            target_proba,
            threshold,
            "CICIDS_to_UNSW",
            rep_name,
            model_name,
        )
        row["train_seconds"] = train_seconds
        reverse_experiments.append(row)

        print(
            model_name,
            "threshold=", round(threshold, 3),
            "F1=", round(row["f1"], 4),
            "PR-AUC=", round(row["pr_auc"], 4),
            "time(s)=", round(train_seconds, 1),
        )

res_reverse = pd.DataFrame(reverse_experiments)
display(res_reverse.sort_values("f1", ascending=False))


## 11. Compare all cross-dataset baselines

The important comparison is not simply the highest raw accuracy. We want the configuration that gives the best **cross-dataset F1 while keeping FPR and PR-AUC credible**.

In [ ]:
# ============================================================
# 11. Final results + benchmark comparison
# ============================================================

all_results = pd.concat([res_unsw_cic, res_reverse], ignore_index=True)
all_results = all_results.sort_values(["direction", "f1"], ascending=[True, False])

display(all_results)

all_results.to_csv(
    TABLES / "03_cross_dataset_strong_baselines.csv",
    index=False,
)

# Current historical benchmark from Phase 12 for context only.
# This is NOT treated as an apples-to-apples comparison unless the split is identical.
HISTORICAL_PHASE12_BENCHMARK_F1 = 0.3385

best_unsw_cic = res_unsw_cic.sort_values("f1", ascending=False).iloc[0]
best_reverse = res_reverse.sort_values("f1", ascending=False).iloc[0]

summary = pd.DataFrame([
    {
        "direction": "UNSW_to_CICIDS",
        "best_representation": best_unsw_cic["representation"],
        "best_model": best_unsw_cic["model"],
        "best_f1": best_unsw_cic["f1"],
        "best_pr_auc": best_unsw_cic["pr_auc"],
        "best_roc_auc": best_unsw_cic["roc_auc"],
        "fpr": best_unsw_cic["fpr"],
        "accuracy": best_unsw_cic["accuracy"],
    },
    {
        "direction": "CICIDS_to_UNSW",
        "best_representation": best_reverse["representation"],
        "best_model": best_reverse["model"],
        "best_f1": best_reverse["f1"],
        "best_pr_auc": best_reverse["pr_auc"],
        "best_roc_auc": best_reverse["roc_auc"],
        "fpr": best_reverse["fpr"],
        "accuracy": best_reverse["accuracy"],
    },
])

display(summary)
summary.to_csv(TABLES / "04_best_model_summary.csv", index=False)

config = {
    "random_state": RANDOM_STATE,
    "cicids_total_rows": TOTAL_CICIDS_ROWS,
    "cicids_sample_n": int(len(cicids_sample)),
    "cicids_eval_n": int(len(cic_eval_idx)),
    "cicids_source_fit_n": int(len(cic_fit_idx)),
    "cicids_source_val_n": int(len(cic_val_idx)),
    "compact_features": len(COMMON_11),
    "expanded_features": int(X_unsw_train_exp.shape[1]),
    "historical_phase12_f1_context": HISTORICAL_PHASE12_BENCHMARK_F1,
    "target_labels_used_for_model_selection": False,
    "target_labels_used_for_threshold_selection": False,
    "cicids_full_dataframe_loaded": False,
}

with open(TABLES / "05_experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\nSaved Phase 16 fixed results to:", TABLES)


## 12. Decision checkpoint

### How to interpret this phase

1. **If a model/representation beats the previous cross-dataset F1 benchmark**, use it as the new base for adaptation. Do not immediately add more features.
2. If the richer representation improves PR-AUC/ROC-AUC but not F1, that means the representation is learning useful target ranking but the decision boundary is still miscalibrated; calibration/prior correction becomes the next targeted experiment.
3. If ExtraTrees or HistGradientBoosting clearly beats Random Forest, use the stronger model in the next adaptation experiment.
4. If all models remain weak, inspect the exact source/target feature distributions and the split before adding another algorithm.
5. Do **not** call the historical Phase 12 number an improvement/regression unless the evaluation population is identical. This notebook creates a reproducible frozen CICIDS evaluation sample for fair comparison among the Phase 16 candidates.

### Important
This phase is intentionally **not** another domain-adaptation trick. It establishes whether the problem is caused by:
- the compact semantic representation;
- model choice;
- insufficient source training data;
- decision threshold/calibration;
- or genuine dataset shift.

That gives us a much stronger foundation for both **performance** and **novelty** in the final project.